[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/blockXblock/berkeley-housing-analysis/blob/main/03_analysis/C4_quality_checks.ipynb)

# C4: Data Quality Checks and Sanity Tests

---

## Learning Objectives

By the end of this notebook, you will be able to:
1. **Check for common data issues** - nulls, duplicates, invalid values
2. **Implement "soft asserts"** - flag problems without crashing
3. **Validate aggregates** - compare totals against expected values
4. **Create a quality report** - summarize data health

## Why Data Quality Matters

In civic data, errors propagate:
- Bad permit data → wrong APR report → state compliance issues
- Missing addresses → geocoding failures → projects missing from maps
- Duplicate records → inflated housing counts → misleading statistics

**Quality checks are not optional** - they're essential infrastructure.

---

## 1. Setup

In [ ]:
import sys
import json
import pandas as pd
import numpy as np
from pathlib import Path
from datetime import datetime

# Find project root
def find_project_root():
    current = Path.cwd()
    for path in [current] + list(current.parents):
        if (path / '00_config').exists() and (path / 'modules').exists():
            return path
    return current

ROOT = find_project_root()
sys.path.insert(0, str(ROOT))

# Load config
config_path = ROOT / '00_config/berkeley_config.json'
if config_path.exists():
    with open(config_path) as f:
        CONFIG = json.load(f)
    DATA_DIR = ROOT / CONFIG['paths']['data_dir']
else:
    DATA_DIR = ROOT / 'data/processed'

print(f"Project root: {ROOT}")
print(f"Data directory: {DATA_DIR}")

In [ ]:
# Load the main dataset
df = pd.read_csv(DATA_DIR / 'housing_projects_FINAL.csv')
print(f"Loaded {len(df)} projects")

## 2. Soft Asserts Pattern

A "soft assert" checks a condition and logs a warning if it fails, but **doesn't crash** the notebook. This lets you see all issues at once.

In [ ]:
# Quality check results collector
quality_issues = []

def soft_assert(condition, message, severity='WARNING'):
    """
    Check a condition and log if it fails.
    
    Parameters:
        condition: Boolean - True if check passes
        message: String describing the check
        severity: 'WARNING', 'ERROR', or 'INFO'
    
    Returns:
        True if condition passed, False otherwise
    """
    if condition:
        print(f"  [PASS] {message}")
        return True
    else:
        print(f"  [{severity}] {message}")
        quality_issues.append({
            'severity': severity,
            'message': message,
            'timestamp': datetime.now().isoformat()
        })
        return False

print("Soft assert function ready.")

## 3. Basic Data Quality Checks

In [ ]:
print("=" * 60)
print("CHECK: Missing Values (Nulls)")
print("=" * 60)

# Critical columns that should never be null
critical_columns = ['address_display', 'net_units', 'status']

for col in critical_columns:
    if col in df.columns:
        null_count = df[col].isna().sum()
        null_pct = 100 * null_count / len(df)
        soft_assert(
            null_count == 0,
            f"{col}: {null_count} nulls ({null_pct:.1f}%)",
            'ERROR' if null_pct > 5 else 'WARNING'
        )
    else:
        soft_assert(False, f"{col}: column not found", 'ERROR')

In [ ]:
print("\n" + "=" * 60)
print("CHECK: Duplicate Records")
print("=" * 60)

# Check for duplicate project IDs
if 'id' in df.columns:
    dup_ids = df['id'].duplicated().sum()
    soft_assert(dup_ids == 0, f"Duplicate project IDs: {dup_ids}")

# Check for duplicate addresses
if 'address_display' in df.columns:
    dup_addr = df['address_display'].duplicated().sum()
    soft_assert(
        dup_addr == 0,
        f"Duplicate addresses: {dup_addr}",
        'WARNING'  # Duplicates might be valid (multiple permits)
    )
    
    if dup_addr > 0:
        print("\n  Duplicated addresses:")
        dups = df[df['address_display'].duplicated(keep=False)]
        for addr in dups['address_display'].unique()[:5]:
            print(f"    - {addr}")

In [ ]:
print("\n" + "=" * 60)
print("CHECK: Value Ranges")
print("=" * 60)

# Units should be positive
if 'net_units' in df.columns:
    negative_units = (df['net_units'] < 0).sum()
    soft_assert(
        negative_units == 0,
        f"Negative unit counts: {negative_units}",
        'ERROR'
    )
    
    zero_units = (df['net_units'] == 0).sum()
    soft_assert(
        zero_units == 0,
        f"Zero unit projects: {zero_units}",
        'WARNING'
    )
    
    # Sanity check: no single project should have > 1000 units
    huge_projects = (df['net_units'] > 1000).sum()
    soft_assert(
        huge_projects == 0,
        f"Projects with >1000 units (verify): {huge_projects}",
        'WARNING'
    )

# Year should be reasonable
if 'year' in df.columns:
    current_year = datetime.now().year
    future_years = (df['year'] > current_year).sum()
    soft_assert(
        future_years == 0,
        f"Projects with future years: {future_years}",
        'WARNING'
    )
    
    old_years = (df['year'] < 2010).sum()
    soft_assert(
        old_years == 0,
        f"Projects before 2010 (verify): {old_years}",
        'INFO'
    )

In [ ]:
print("\n" + "=" * 60)
print("CHECK: APN Coverage")
print("=" * 60)

if 'apn' in df.columns:
    has_apn = df['apn'].notna() & (df['apn'] != '') & (df['apn'] != 'nan')
    apn_coverage = 100 * has_apn.sum() / len(df)
    
    soft_assert(
        apn_coverage >= 95,
        f"APN coverage: {apn_coverage:.1f}% ({has_apn.sum()}/{len(df)})",
        'WARNING' if apn_coverage >= 90 else 'ERROR'
    )

In [ ]:
print("\n" + "=" * 60)
print("CHECK: Geocoding Coverage")
print("=" * 60)

if 'latitude' in df.columns and 'longitude' in df.columns:
    has_coords = df['latitude'].notna() & df['longitude'].notna()
    geo_coverage = 100 * has_coords.sum() / len(df)
    
    soft_assert(
        geo_coverage >= 99,
        f"Geocoding coverage: {geo_coverage:.1f}%",
        'WARNING' if geo_coverage >= 95 else 'ERROR'
    )
    
    # Check if coordinates are in Berkeley bounds
    BERKELEY_BOUNDS = {
        'lat_min': 37.84, 'lat_max': 37.92,
        'lon_min': -122.33, 'lon_max': -122.22
    }
    
    in_bounds = (
        (df['latitude'] >= BERKELEY_BOUNDS['lat_min']) &
        (df['latitude'] <= BERKELEY_BOUNDS['lat_max']) &
        (df['longitude'] >= BERKELEY_BOUNDS['lon_min']) &
        (df['longitude'] <= BERKELEY_BOUNDS['lon_max'])
    )
    
    out_of_bounds = has_coords.sum() - in_bounds.sum()
    soft_assert(
        out_of_bounds == 0,
        f"Projects outside Berkeley bounds: {out_of_bounds}",
        'WARNING'
    )

## 4. Aggregate Validation

Compare computed totals against known/expected values.

In [ ]:
print("\n" + "=" * 60)
print("CHECK: Aggregate Validation")
print("=" * 60)

# Expected values (update these based on known data)
EXPECTED = {
    'min_projects': 100,      # We expect at least 100 projects
    'max_projects': 200,      # But probably not more than 200
    'min_total_units': 4000,  # At least 4000 units total
    'max_total_units': 8000,  # But not more than 8000
}

# Check project count
soft_assert(
    len(df) >= EXPECTED['min_projects'],
    f"Project count ({len(df)}) >= expected minimum ({EXPECTED['min_projects']})"
)

soft_assert(
    len(df) <= EXPECTED['max_projects'],
    f"Project count ({len(df)}) <= expected maximum ({EXPECTED['max_projects']})",
    'WARNING'
)

# Check total units
if 'net_units' in df.columns:
    total_units = df['net_units'].sum()
    
    soft_assert(
        total_units >= EXPECTED['min_total_units'],
        f"Total units ({total_units:,.0f}) >= expected minimum ({EXPECTED['min_total_units']:,})"
    )
    
    soft_assert(
        total_units <= EXPECTED['max_total_units'],
        f"Total units ({total_units:,.0f}) <= expected maximum ({EXPECTED['max_total_units']:,})",
        'WARNING'
    )

In [ ]:
print("\n" + "=" * 60)
print("CHECK: Status Distribution")
print("=" * 60)

if 'status' in df.columns:
    status_counts = df['status'].value_counts()
    
    print("\nStatus distribution:")
    for status, count in status_counts.items():
        pct = 100 * count / len(df)
        print(f"  {status:30} {count:>4} ({pct:>5.1f}%)")
    
    # Check for unknown/empty statuses
    unknown_statuses = df['status'].isna().sum()
    soft_assert(
        unknown_statuses == 0,
        f"Unknown/empty statuses: {unknown_statuses}"
    )

## 5. Quality Report Summary

In [ ]:
print("\n" + "=" * 60)
print("DATA QUALITY REPORT")
print("=" * 60)
print(f"\nGenerated: {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}")
print(f"Dataset: housing_projects_FINAL.csv")
print(f"Records: {len(df)}")

# Count issues by severity
errors = len([i for i in quality_issues if i['severity'] == 'ERROR'])
warnings = len([i for i in quality_issues if i['severity'] == 'WARNING'])
info = len([i for i in quality_issues if i['severity'] == 'INFO'])

print(f"\nIssues Found:")
print(f"  Errors:   {errors}")
print(f"  Warnings: {warnings}")
print(f"  Info:     {info}")

if errors == 0 and warnings == 0:
    print(f"\nStatus: ALL CHECKS PASSED")
elif errors == 0:
    print(f"\nStatus: PASSED WITH WARNINGS")
else:
    print(f"\nStatus: ISSUES NEED ATTENTION")

# List all issues
if quality_issues:
    print(f"\nDetailed Issues:")
    for issue in quality_issues:
        print(f"  [{issue['severity']}] {issue['message']}")

In [ ]:
# Optionally save report to file
OUTPUT_DIR = ROOT / 'data/outputs'
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

report_path = OUTPUT_DIR / 'quality_report.json'

report = {
    'generated': datetime.now().isoformat(),
    'dataset': 'housing_projects_FINAL.csv',
    'record_count': len(df),
    'error_count': errors,
    'warning_count': warnings,
    'issues': quality_issues
}

with open(report_path, 'w') as f:
    json.dump(report, f, indent=2)

print(f"\nReport saved to: {report_path}")

---

## Summary

This notebook taught you to:

1. **Use soft asserts** - check conditions without crashing
2. **Check for common issues** - nulls, duplicates, invalid values
3. **Validate aggregates** - compare totals against expectations
4. **Generate quality reports** - document data health

### Key Pattern
```python
def soft_assert(condition, message, severity='WARNING'):
    if condition:
        print(f"[PASS] {message}")
    else:
        print(f"[{severity}] {message}")
        quality_issues.append(...)
```

### When to Run Quality Checks
- **Before analysis** - ensure data is clean
- **After imports** - verify data loaded correctly
- **Before publishing** - catch issues before they go public
- **Regularly** - as part of automated monitoring

**Next:** See `D3_alerts_monitoring.ipynb` for ongoing monitoring patterns.